In [29]:
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)

# Returns comparison

In [30]:
returns_10y = pd.read_parquet("data/returns_10y.parquet")
returns_5y  = pd.read_parquet("data/returns_5y.parquet")

In [31]:
mu_5y = returns_5y.mean().values.to_numpy()     
mu_10y = returns_10y.mean().values.to_numpy()   

In [32]:
with open("results/optimized_weights_10y.pkl", "rb") as f:
    results_10y = pickle.load(f)

with open("results/optimized_weights_5y.pkl", "rb") as f:
    results_5y = pickle.load(f)


In [33]:
results_10y

[{'Volatility': 0.1600000055941139,
  'ESG': 86.15021703036956,
  'Weights': array([ 1.83619014e-09,  6.19814041e-09,  3.08837532e-10,  2.62564436e-09,
          4.20640798e-09,  2.42069802e-10,  1.08716531e-08,  5.75222536e-10,
          1.14742287e-09,  2.54945240e-08,  3.64113194e-10,  1.05628383e-09,
          2.08334985e-09,  2.67509474e-02,  4.20417505e-01,  6.90706534e-09,
          2.39574970e-09,  2.04120064e-09,  2.28893863e-01,  1.00422620e-07,
          8.43848889e-02,  1.30118758e-09,  1.92713658e-10,  1.39014295e-09,
          4.09019447e-09, -2.04015783e-10,  6.79355013e-09,  8.82890371e-02,
          1.51263576e-01])},
 {'Volatility': 0.1700000106740791,
  'ESG': 87.24856912010102,
  'Weights': array([ 2.64718688e-09,  7.72819444e-09,  7.98948730e-11,  4.32218175e-09,
          5.20663641e-09, -9.37007829e-11,  1.53368825e-08,  2.44053040e-10,
          8.50187203e-10,  1.96479858e-08,  1.60109708e-10,  1.10731911e-09,
          2.19623705e-09,  4.04185567e-02,  4.00262

In [34]:
for r in results_5y:
    w = r["Weights"]
    r["ExpReturn"] = (mu_5y @ w) * 252

for r in results_10y:
    w = r["Weights"]
    r["ExpReturn"] = (mu_10y @ w) * 252

comparison_5y = pd.DataFrame({
    "Volatility": [r["Volatility"] for r in results_5y],
    "ESG": [r["ESG"] for r in results_5y],
    "ExpectedReturn": [r["ExpReturn"] for r in results_5y]
})

comparison_10y = pd.DataFrame({
    "Volatility": [r["Volatility"] for r in results_10y],
    "ESG": [r["ESG"] for r in results_10y],
    "ExpectedReturn": [r["ExpReturn"] for r in results_10y]
})


In [35]:
comparison_5y

,Volatility,ESG,ExpectedReturn
0,0.160000,81.577382,0.036180
1,0.170000,84.534386,0.041116
2,0.180000,86.083657,0.023036
3,0.190000,86.991575,0.006653
4,0.200000,87.645204,-0.012250
5,0.210000,88.091113,-0.028795
6,0.220000,88.456178,-0.042336
7,0.230000,88.778332,-0.054277
8,0.240000,89.073283,-0.065231
9,0.250000,89.316692,-0.101288


In [36]:
comparison_10y

,Volatility,ESG,ExpectedReturn
0,0.16000,86.150217,0.041737
1,0.17000,87.248569,0.032425
2,0.18000,87.978642,0.019884
3,0.19000,88.475899,0.008864
4,0.20000,88.886053,-0.000228
5,0.21000,89.245562,-0.014648
6,0.22000,89.486087,-0.040936
7,0.23000,89.626225,-0.050983
8,0.24000,89.732821,-0.058618
9,0.25000,89.822700,-0.062990


# Weights comparison

In [37]:
def weight_concentration(w):
    return np.sum(w**2)

for r in results_5y:
    r["Concentration"] = weight_concentration(r["Weights"])

for r in results_10y:
    r["Concentration"] = weight_concentration(r["Weights"])


In [38]:
vol_levels = [0.16, 0.20, 0.25, 0.30]

In [39]:
w_bench = pd.read_csv("data/benchmark_weights.csv")

# set Instrument as index
w_bench = w_bench.set_index("Instrument")
# keep only assets in returns (important!)
w_bench_aligned = w_bench.loc[returns_10y.columns, "BenchWeight"]


In [40]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

sigma_grid = np.linspace(0.16, 0.35, 20)
idxes =[]
for ax, vol in zip(axes, vol_levels):
    # index based on TARGET volatility
    idx = np.argmin(np.abs(sigma_grid - vol))
    idxes.append(idx)
    r5 = results_5y[idx]
    r10 = results_10y[idx]

    w5 = r5["Weights"]
    w10 = r10["Weights"]

    df = pd.DataFrame(
    {
        "5Y": w5,
        "10Y": w10
    },
    index=returns_10y.columns
)


    top = df.sort_values("10Y", ascending=False)

    x = np.arange(len(top))
    width = 0.35


    ax.bar(x - width, top["5Y"], width, label="5Y")
    ax.bar(x,          top["10Y"], width, label="10Y")


    ax.set_title(f"Target volatility ≈ {vol:.0%}")
    ax.set_xticks(x)
    ax.set_xticklabels(top.index, rotation=45, ha="right")
    ax.grid(axis="y")

    # ---- concentration annotation ONLY ----
    textstr = (
        f"Concentration (HHI):\n"
        f"5Y: {r5['Concentration']:.3f}\n"
        f"10Y: {r10['Concentration']:.3f}"
    )

    ax.text(
    0.98, 0.98,
    textstr,
    transform=ax.transAxes,
    fontsize=9,
    horizontalalignment="right",
    verticalalignment="top",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
)

axes[0].legend()
plt.tight_layout()
plt.savefig("figures/weights_comparison_multiple_vols.pdf")
plt.savefig("figures/weights_comparison_multiple_vols.png")
plt.close()


In [41]:
import matplotlib.pyplot as plt
import numpy as np

# align benchmark weights to returns universe
w_bench_aligned = w_bench.loc[returns_10y.columns, "BenchWeight"]
w_bench_aligned = w_bench_aligned / w_bench_aligned.sum()

# benchmark concentration (Herfindahl index)
bench_concentration = np.sum(w_bench_aligned.values**2)

# sort for readability
w_bench_sorted = w_bench_aligned.sort_values(ascending=False)

plt.figure(figsize=(10, 4))
plt.bar(w_bench_sorted.index, w_bench_sorted.values)

plt.xticks(rotation=45, ha="right")
plt.ylabel("Benchmark Weight")
plt.title("Benchmark Portfolio Weights (DJIA, Market-Cap Weighted)")
plt.grid(axis="y")

# ---- concentration annotation ----
plt.text(
    0.98, 0.95,
    f"Concentration (HHI): {bench_concentration:.3f}",
    transform=plt.gca().transAxes,
    ha="right",
    va="top",
    fontsize=10,
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
)

plt.tight_layout()
plt.savefig("figures/benchmark_weights.pdf")
plt.savefig("figures/benchmark_weights.png")
plt.close()


In [42]:
idxes

[0, 4, 9, 14]

In [50]:
comparison_10y['Volatility'] = comparison_10y['Volatility'].round(2)
comparison_5y['Volatility'] = comparison_5y['Volatility'].round(2)

In [52]:
pd.merge(comparison_5y.iloc[idxes].drop(columns='ESG'), comparison_10y.iloc[idxes].drop(columns='ESG'), how='left', on='Volatility').to_latex()

'\\begin{tabular}{lrrr}\n\\toprule\n & Volatility & ExpectedReturn_x & ExpectedReturn_y \\\\\n\\midrule\n0 & 0.160000 & 0.036180 & 0.041737 \\\\\n1 & 0.200000 & -0.012250 & -0.000228 \\\\\n2 & 0.250000 & -0.101288 & -0.062990 \\\\\n3 & 0.300000 & -0.172680 & -0.078561 \\\\\n\\bottomrule\n\\end{tabular}\n'

In [54]:
bench_exp_return_is_5y = (w_bench_aligned.values @ mu_5y) * 252
bench_exp_return_is_10y = (w_bench_aligned.values @ mu_10y) * 252
print(bench_exp_return_is_5y, bench_exp_return_is_10y)

0.18539828638275482 0.15626010291033715
